In [ ]:
import logging
import csv
from typing import Any, Dict, List
import json
import json
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import difflib

from kinexon_handball_api.handball import HandballAPI

In [ ]:
load_dotenv()  # Load environment variables from .env file if present


In [ ]:
# connect duckdb
import duckdb
con = duckdb.connect(database='../data/mydb2024-25.duckdb', read_only=False)

In [ ]:
api = HandballAPI(
    base_url=os.getenv(
        "ENDPOINT_KINEXON_SESSION", "https://hbl-cloud.kinexon.com/api"
    ),
    api_key=os.getenv("API_KEY_KINEXON", "your_api_key_here"),
    username_basic=os.getenv("USERNAME_KINEXON_SESSION", "your_username_here"),
    password_basic=os.getenv("PASSWORD_KINEXON_SESSION", "your_password_here"),
    username_main=os.getenv("USERNAME_KINEXON_MAIN", "your_username_here"),
    password_main=os.getenv("PASSWORD_KINEXON_MAIN", "your_password_here"),
    endpoint_session=os.getenv(
        "ENDPOINT_KINEXON_SESSION",
        "https://hbl-cloud.kinexon.com/api/session",
    ),
    endpoint_main=os.getenv(
        "ENDPOINT_KINEXON_MAIN",
        "https://hbl-cloud.kinexon.com/api",
    ),
    timeout=10000,
)

In [ ]:
avail = api.get_available_metrics_and_events()
# print("Available metrics and events:", len(avail))

ids_team = api.fetch_team_ids("2024-25")

print("Team IDs:", ids_team)

In [ ]:
# select * from table teams
df_teams_sportradar = con.execute("SELECT * FROM teams").df()
print("teams")
display(df_teams_sportradar.head())

# select * from table fixtures

df_fixtures_sportradar = con.execute("SELECT * FROM fixtures").df()
print("Fixtures")
display(df_fixtures_sportradar.head())

## Iterate of Sportradar fixtures to get date of game and therefore sessionid of kinexon.

In [ ]:
import datetime
import pandas as pd

# Fast lookup for team id with fuzzy matching
def find_best_team_match(target_name: str, team_list: list, threshold: float = 0.8):
    """Find best matching team using fuzzy string matching"""
    best_match = None
    best_score = 0
    best_id = None
    
    for team in team_list:
        team_name = team["name"]
        # Calculate similarity ratio
        similarity = difflib.SequenceMatcher(None, target_name.lower(), team_name.lower()).ratio()
        
        if similarity > best_score and similarity >= threshold:
            best_score = similarity
            best_match = team_name
            best_id = team["id"]
    
    return best_id, best_match, best_score

results = []     # successful matches
failures = []    # failures with reasons

print(f"Processing {len(df_fixtures_sportradar)} fixtures…")

for index, row_sportradar in df_fixtures_sportradar.iterrows():
    fixture_id   = row_sportradar['fixtureId']
    name_local   = row_sportradar['nameLocal']
    sr_external_id = row_sportradar['externalId']
    competitors  = row_sportradar['competitors']

    # home team
    home_list = [comp for comp in competitors if comp['isHome']]
    if not home_list:
        print(f"[{fixture_id}] ✖ no home competitor")
        failures.append({"fixtureId": fixture_id, "reason": "no_home_competitor"})
        continue
    name_team_home = home_list[0]['nameFullLocal']
    # fix HSV Hamburg naming inconsistency
    if name_team_home == "Handball Sport Verein Hamburg":
        name_team_home = "HSV Hamburg"

    # map to team id from ids_team using fuzzy matching
    id_team_home, matched_name, similarity_score = find_best_team_match(name_team_home, ids_team)
    if id_team_home is None:
        print(f"[{fixture_id}] {name_local} ✖ team '{name_team_home}' not found in ids_team (best similarity < 0.8)")
        failures.append({"fixtureId": fixture_id, "reason": "team_not_found_fuzzy", "team": name_team_home})
        continue
    elif matched_name != name_team_home:
        print(f"[{fixture_id}] ≈ fuzzy matched '{name_team_home}' → '{matched_name}' (similarity: {similarity_score:.2f})")

    # day window (local 00:00–24:00) exactly as in your code
    date_game_start = pd.to_datetime(row_sportradar['startTimeLocal'])
    date_game_start = date_game_start.replace(hour=0, minute=0, second=0, microsecond=0)
    date_game_end = date_game_start.date() + pd.Timedelta(hours=24)

    datetime_start = datetime.datetime.fromisoformat(str(date_game_start))
    datetime_end   = datetime.datetime.fromisoformat(str(date_game_end))

    sessions = api.get_sessions_for_team(id_team_home, start=datetime_start, end=datetime_end)
    if len(sessions) == 0:
        print(f"[{fixture_id}] ✖ no sessions on {date_game_start.date()} for '{name_team_home}'")
        failures.append({"fixtureId": fixture_id, "reason": "no_sessions", "team": name_team_home})
        continue

    # find first session whose group_names contains the home team (using matched name for comparison)
    found = False
    for session in sessions:
        df_session = pd.DataFrame([session.to_dict()])
        group_names = df_session['group_names'].values[0]
        # Try both original name and matched name
        if name_team_home in group_names or matched_name in group_names:
            sid = df_session.get('session_id', pd.Series([None])).values[0]
            results.append({
                "fixtureId": fixture_id,
                "nameLocal": name_local,
                "homeTeam": name_team_home,
                "matchedTeam": matched_name,
                "teamId": id_team_home,
                "session_id": sid,
                "sr_external_id": sr_external_id,
                "similarity_score": similarity_score,
            })
            print(f"[{fixture_id}] ✓ synced → session {sid} ({matched_name})")
            found = True
            break

    if not found:
        print(f"[{fixture_id}] ✖ sessions found but none matched group_names for '{name_team_home}' or '{matched_name}'")
        failures.append({"fixtureId": fixture_id, "reason": "no_matching_group", "team": name_team_home, "matched_team": matched_name})

# --- summary stats ---
total = len(df_fixtures_sportradar)
ok = len(results)
fail = len(failures)
pct = (ok / total * 100.0) if total else 0.0

print("\n=== Sync Summary ===")
print(f"Total fixtures:   {total}")
print(f"Synced (matched): {ok}")
print(f"Failed:           {fail}")
print(f"Success rate:     {pct:.1f}%")

df_synced = pd.DataFrame(results).sort_values("fixtureId") if results else pd.DataFrame(columns=["fixtureId","session_id"])
df_failed = pd.DataFrame(failures).sort_values("fixtureId") if failures else pd.DataFrame(columns=["fixtureId","reason"])

display(df_synced.head(10))
display(df_failed.head(10))


In [ ]:
# --- persist session_id to DuckDB and write a run log ---

# 1) Ensure target column exists
con.execute("ALTER TABLE fixtures ADD COLUMN IF NOT EXISTS session_id BIGINT")

# 2) Update fixtures.session_id for matched rows
if not df_synced.empty:
    df_updates = (
        df_synced[["fixtureId", "session_id"]]
        .dropna(subset=["session_id"])
        .astype({"fixtureId": "string"})
    )
    con.register("df_updates", df_updates)
    con.execute("""
        UPDATE fixtures AS f
        SET session_id = u.session_id
        FROM df_updates AS u
        WHERE f.fixtureId::STRING = u.fixtureId
    """)
    con.unregister("df_updates")

# 3) Build a normalized run log like the ETL and persist it
import pandas as pd

df_log_ok = df_synced.copy()
if not df_log_ok.empty:
    df_log_ok["status"] = "ok"
    df_log_ok["team"] = df_log_ok.get("homeTeam", None)
    df_log_ok["matched_team"] = df_log_ok.get("matchedTeam", None)
    df_log_ok["reason"] = None

df_log_fail = pd.DataFrame(df_failed).copy()
if not df_log_fail.empty:
    df_log_fail["status"] = "fail"
    # align columns
    for col in ["session_id","team","matched_team","similarity_score","nameLocal","sr_external_id"]:
        if col not in df_log_fail.columns:
            df_log_fail[col] = None

# unify schema
cols = ["fixtureId","status","reason","session_id","team","matched_team","similarity_score"]
df_log = pd.concat([df_log_ok[cols], df_log_fail[cols]], ignore_index=True) if (not df_log_ok.empty or not df_log_fail.empty) else pd.DataFrame(columns=cols)

if not df_log.empty:
    df_log = df_log.assign(run_ts=pd.Timestamp.utcnow())
    # persist
    con.execute("DROP TABLE IF EXISTS fixtures_session_sync_log")
    con.execute("""
        CREATE TABLE fixtures_session_sync_log (
            run_ts TIMESTAMP,
            fixtureId TEXT,
            status VARCHAR,
            reason VARCHAR,
            session_id BIGINT,
            team VARCHAR,
            matched_team VARCHAR,
            similarity_score DOUBLE
        )
    """)
    con.register("df_log", df_log)
    con.execute("""
        INSERT INTO fixtures_session_sync_log
        SELECT run_ts, fixtureId::TEXT, status, reason, session_id, team, matched_team, similarity_score
        FROM df_log
    """)
    con.unregister("df_log")

print("Persisted session_ids and wrote fixtures_session_sync_log.")
